In [1]:
# converting EMIT mineral mass fraction to geogrid specific binary format

In [2]:
import os
import numpy as np
import xarray as xr

In [3]:
input_file = "EMIT_L3_ASA_002.nc" 
ds = xr.open_dataset(input_file)

In [4]:
out_root = "EMIT_geog"

In [5]:
precision = 0.001
bytes_per_element = 2
endian = ">"
fill_value = 0

In [6]:
mineral_map = {
    "Calcite": "emit_calcite",
    "Chlorite": "emit_chlorite",
    "Dolomite": "emit_dolomite",
    "Goethite": "emit_goethite",
    "Gypsum": "emit_gypsum",
    "Hematite": "emit_hematite",
    "Illite+Muscovite": "emit_illite_muscovite",
    "Kaolinite": "emit_kaolinite",
    "Montmorillonite": "emit_montmorillonite",
    "Vermiculite": "emit_vermiculite",
    "Quartz+Feldspar": "emit_quartz_feldspar",}

In [7]:
lat = ds["lat"]
lon = ds["lon"]

n_rows = len(lat)   # 220
n_cols = len(lon)   # 690

In [8]:

def convert_to_geogrid_int(array, precision=0.001, bytes_per_element=2, fill_value=0):

    arr = np.array(array, dtype=np.float64)

    # replace NaN
    arr = np.nan_to_num(arr, nan=fill_value, posinf=fill_value, neginf=fill_value)

    # scale
    scaled = np.rint(arr / precision).astype(np.int64)

    # handle negatives (not needed here but kept safe)
    shift_value = 2**(8 * bytes_per_element)
    scaled[scaled < 0] += shift_value

    return scaled



In [9]:

def write_index(out_dir, mineral_name):

    text = f"""type=continuous
projection=regular_ll
dx=0.5
dy=0.5
known_x=1.0
known_y=1.0
known_lat=-54.5
known_lon=-165.0
wordsize=2
tile_x=690
tile_y=220
tile_z=1
endian=big
signed=no
units="fraction"
description="EMIT {mineral_name} mineral fraction"
"""

    with open(os.path.join(out_dir, "index"), "w") as f:
        f.write(text)

In [11]:

xstart = 1
xend = n_cols
ystart = 1
yend = n_rows

tile_name = f"{xstart:05d}-{xend:05d}.{ystart:05d}-{yend:05d}"

for varname, folder in mineral_map.items():

    print(f"\nProcessing {varname}")

    if varname not in ds:
        print("  Not found, skipping")
        continue

    data = ds[varname].values

    # convert
    int_data = convert_to_geogrid_int(data, precision, bytes_per_element)

    # flip south first
    int_data = np.flipud(int_data)

    # create folder
    out_dir = os.path.join(out_root, folder)
    os.makedirs(out_dir, exist_ok=True)

    # write binary
    out_file = os.path.join(out_dir, tile_name)
    int_data.astype(f"{endian}i{bytes_per_element}").tofile(out_file)

    # write index
    write_index(out_dir, varname)

    print(f"  Binary → {out_file}")
    print(f"  Index  → {out_dir}/index")




Processing Calcite
  Binary → EMIT_geog\emit_calcite\00001-00690.00001-00220
  Index  → EMIT_geog\emit_calcite/index

Processing Chlorite
  Binary → EMIT_geog\emit_chlorite\00001-00690.00001-00220
  Index  → EMIT_geog\emit_chlorite/index

Processing Dolomite
  Binary → EMIT_geog\emit_dolomite\00001-00690.00001-00220
  Index  → EMIT_geog\emit_dolomite/index

Processing Goethite
  Binary → EMIT_geog\emit_goethite\00001-00690.00001-00220
  Index  → EMIT_geog\emit_goethite/index

Processing Gypsum
  Binary → EMIT_geog\emit_gypsum\00001-00690.00001-00220
  Index  → EMIT_geog\emit_gypsum/index

Processing Hematite
  Binary → EMIT_geog\emit_hematite\00001-00690.00001-00220
  Index  → EMIT_geog\emit_hematite/index

Processing Illite+Muscovite
  Binary → EMIT_geog\emit_illite_muscovite\00001-00690.00001-00220
  Index  → EMIT_geog\emit_illite_muscovite/index

Processing Kaolinite
  Binary → EMIT_geog\emit_kaolinite\00001-00690.00001-00220
  Index  → EMIT_geog\emit_kaolinite/index

Processing Mo